# G02-B v4 — Lesion-aware N01: HAM10000 / BCN20000 source evaluation

Kaggle GPU + Internet ON. Attach completed Kaggle outputs: **G01**, **R04** (`lesion-r04-ssl-classification-s42`), **R06** (`lesion-r06-n01-rotation-simclr-feature-fusion-s42`); R05 optional. Exact N01 processed images from the previous N01 Kaggle output are preferred; the official 2019 ZIP is downloaded if they are not available. Six frozen prediction sources. Strict R04 input-reproduction audit and R06 feature-normalization replay. Default `validation` cohort = 2,109 HAM+BCN images. **Locked N01 test untouched** unless explicit `G02B_COHORT=locked_test` and `G02B_ALLOW_LOCKED_TEST=1` after final ensemble lock.

The 2,186-image N01 validation set includes other sources; this study uses only its 2,109 HAM/BCN images. No base model or fusion weights are refit.

## 1. Config, G01 cohort, and direct official source download

In [1]:
import os,json,hashlib,zipfile,shutil,subprocess,time,re,io
from pathlib import Path
import pandas as pd,numpy as np
SEED=42;CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
INPUT=Path('/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data')
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
def req(ok,msg):
    if not bool(ok):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def writejson(p,x):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(x,indent=2,allow_nan=False,default=str));tmp.replace(p)
def find_unique(glob_name,root=INPUT,optional=False):
    hits=sorted(p for p in root.rglob(glob_name) if p.is_file())
    if not hits and optional:return None
    req(len(hits)==1,f'Expected exactly one {glob_name}; got {len(hits)}: {hits[:5]}')
    return hits[0]
def download(url,out,min_size=100):
    out=Path(out);out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and out.stat().st_size>=min_size:return out
    cmd=['curl','--fail','--location','--retry','6','--retry-delay','5','--connect-timeout','30','--continue-at','-','--output',str(out),url]
    r=subprocess.run(cmd,check=False)
    req(r.returncode==0 and out.is_file() and out.stat().st_size>=min_size,f'Download failed ({r.returncode}): {url}')
    return out
def getzip():
    # Reuse a prior Kaggle notebook output, otherwise download the official ISIC 2019 archive.
    zps=sorted([p for p in INPUT.rglob('ISIC_2019_Training_Input.zip') if p.is_file()])
    zps+=sorted([p for p in (WORK/'g_cache').glob('ISIC_2019_Training_Input.zip') if p.is_file()])
    if zps:archive=zps[0]
    else:
        req(os.getenv('KAGGLE_INTERNET_ENABLED','1')=='1','Enable Kaggle Internet for first download')
        archive=download('https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip',WORK/'g_cache/ISIC_2019_Training_Input.zip',min_size=5*10**9)
    with zipfile.ZipFile(archive) as z:
        names={Path(x).stem:x for x in z.namelist() if x.lower().endswith(('.jpg','.jpeg','.png'))}
    req(len(names)==25331,f'Official ISIC 2019 archive should contain 25,331 distinct JPEGs, found {len(names)}')
    return archive,names

import torch
from torch import nn
from torchvision.models import resnet50
from torchvision import transforms
from scipy.special import softmax
from PIL import Image,ImageOps
NAME='g02b-n01-lesion-aware-ham-bcn';OUT=WORK/NAME
for k in ['predictions','reports','tables']:(OUT/k).mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
req(DEVICE.type=='cuda' or os.getenv('G02B_ALLOW_CPU','0')=='1','Enable Kaggle GPU')
C=8;DIM=2048;DROPOUT=.35;BATCH=48;CHUNK=256;MAX_HOURS=8.0;START=time.monotonic()
SOURCES=['rotation','simclr','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
mode=os.getenv('G02B_COHORT','validation');req(mode in ['validation','locked_test'],'Invalid cohort')
if mode=='locked_test':req(os.getenv('G02B_ALLOW_LOCKED_TEST','0')=='1','Locked test requires explicit opt-in; do not rank ensembles on test')
g01_path=find_unique('g01_status.json');G01=g01_path.parent.parent
info=json.loads(g01_path.read_text());req(info['status']=='COMPLETED' and info['protocol_ready'],'G01 incomplete')
cohort=G01/'manifests'/('G02B_validation.csv' if mode=='validation' else 'G02B_locked_test.csv')
df=pd.read_csv(cohort,dtype={'image_id':str}).reset_index(drop=True)
req(len(df)>0 and df.image_id.is_unique and df.n01_split.eq('validation' if mode=='validation' else 'test').all(),'Wrong or duplicate cohort')
req(df.source.isin(['HAM10000','BCN20000']).all(),'Unexpected source outside G01 membership')
req(df.label.isin(CLASSES).all(),'Invalid labels');df['label_id']=df.label.map({c:i for i,c in enumerate(CLASSES)}).astype(int)
# Prefer EXACT N01 resized training images from a prior Kaggle notebook output.
# When they are unavailable, download official ISIC2019 images and reconstruct preprocessing.
N01_PREPARED={};N01_PREPARED_ROOT=None
root_env=os.getenv('G02B_N01_IMAGES_ROOT','').strip()
if root_env:
    root=Path(root_env).resolve();req(root.is_dir(),'G02B_N01_IMAGES_ROOT not a directory')
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in ['.jpg','.jpeg','.png']:
            req(p.stem not in N01_PREPARED,'Duplicate N01 image basename')
            N01_PREPARED[p.stem]=p
    N01_PREPARED_ROOT=root
else:
    for mp in sorted(INPUT.rglob('experiment_master_manifest.csv')):
        root=mp.parent.parent
        if not (root/'images').is_dir():continue
        temp=pd.read_csv(mp,dtype=str,keep_default_na=False)
        if not {'image_id','image_relpath','split'}.issubset(temp.columns):continue
        samples=temp.loc[temp.split.eq('validation')].head(16)
        if len(samples)<8 or not all((root/Path(v)).is_file() for v in samples.image_relpath):continue
        paths={}
        for img,rel in zip(temp.image_id,temp.image_relpath):
            rel=Path(rel)
            if rel.is_absolute() or '..' in rel.parts:raise RuntimeError('Unsafe N01 image path in locked manifest')
            if not (root/rel).is_file():continue
            req(img not in paths,'Duplicate N01 image key');paths[img]=root/rel
        if len(paths)>=23187 and set(df.image_id).issubset(paths):
            req(not N01_PREPARED,'Multiple N01 processed image roots; set G02B_N01_IMAGES_ROOT')
            N01_PREPARED=paths;N01_PREPARED_ROOT=root
if N01_PREPARED and set(df.image_id).issubset(N01_PREPARED):
    archive=None;members={};print('Using verified local N01 processed image source:',N01_PREPARED_ROOT)
else:
    archive,members=getzip()
    req(set(df.image_id).issubset(members),'Images missing from official archive')
    print('N01 processed images not available; will verify raw-image preprocessing against R04 predictions')
print('G02-B',mode,len(df),'sources=',df.source.value_counts().to_dict(),'device=',DEVICE)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9318M  100 9318M    0     0  71.7M      0  0:02:09  0:02:09 --:--:-- 79.7M


N01 processed images not available; will verify raw-image preprocessing against R04 predictions
G02-B validation 2109 sources= {'HAM10000': 1060, 'BCN20000': 1049} device= cuda


## 2. Discover and verify R04/R06 checkpoints and trained encoders

In [2]:
STAGE=WORK/'.g02b_inputs';STAGE.mkdir(exist_ok=True)
def find_stage_roots(base):
    d={'R04':[],'R05':[],'R06':[]}
    for name in ['run_summary.json','final_status.json']:
        for p in base.rglob(name):
            if p.parent.name!='reports':continue
            try:j=json.loads(p.read_text())
            except Exception:continue
            tag=j.get('stage');tag='R04' if name=='run_summary.json' and tag=='R04' else tag
            if tag not in d or j.get('seed')!=42 or j.get('status')!='COMPLETED':continue
            d[tag].append(p.parent.parent.resolve())
    return {k:sorted(set(v)) for k,v in d.items()}
found=find_stage_roots(INPUT)
if len(found['R04'])!=1 or len(found['R06'])!=1:
    for zp in sorted(INPUT.rglob('*.zip')):
        if zp.name=='ISIC_2019_Training_Input.zip':continue
        with zipfile.ZipFile(zp) as z:
            names=z.namelist();relevant=zp.name=='_output_.zip' or any(n.endswith('reports/run_summary.json') or n.endswith('reports/final_status.json') for n in names)
            if not relevant:continue
            req(sum(i.file_size for i in z.infolist())<12*2**30,'Unexpectedly large notebook artifact ZIP')
            dest=STAGE/hashlib.sha256(str(zp).encode()).hexdigest()[:12];dest.mkdir(exist_ok=True)
            for ent in z.infolist():
                if ent.is_dir():continue
                rel=Path(ent.filename.replace(chr(92),'/'));req(not rel.is_absolute() and '..' not in rel.parts,'Unsafe archive member')
                path=(dest/rel).resolve();req(path.is_relative_to(dest.resolve()),'Archive traversal')
                path.parent.mkdir(parents=True,exist_ok=True)
                if path.exists() and path.stat().st_size==ent.file_size:continue
                with z.open(ent) as src,open(path,'wb') as dst:shutil.copyfileobj(src,dst,1<<20)
    more=find_stage_roots(STAGE)
    found={k:sorted(set(found[k]+more[k])) for k in found}
for k in ['R04','R06']:req(len(found[k])==1,f'{k}: expected one completed output, found {found[k]}')
R04=found['R04'][0];R06=found['R06'][0]
run04=json.loads((R04/'reports/run_summary.json').read_text());run06=json.loads((R06/'reports/final_status.json').read_text())
req(run04['n01_manifest_sha256']==run06['n01_manifest_sha256'],'R04 and R06 trained on different N01 manifest')
req(run04['seed']==run06['seed']==42,'Seed disagreement')
if found['R05']:
    req(len(found['R05'])==1,'Ambiguous R05 artifact')
    r05=json.loads((found['R05'][0]/'reports/final_status.json').read_text())
    req(r05['n01_manifest_sha256']==run06['n01_manifest_sha256'],'R05/R06 identity disagreement')
print('Matched checkpoints:',R04,R06)

Matched checkpoints: /kaggle/input/notebooks/sudiptoroy007/lesion-r04-ssl-classification-s42/lesion-r04-ssl-classification-s42 /kaggle/input/notebooks/masumbillah029/lesion-r06-n01-rotation-simclr-feature-fusion-s42/lesion-r06-n01-rotation-simclr-feature-fusion-s42


In [3]:
class R04Classifier(nn.Module):
    def __init__(self,src):
        super().__init__()
        self.encoder=resnet50(weights=None);self.encoder.fc=nn.Identity()
        self.classifier=nn.Linear(2048,8)
        best=torch.load(R04/'models'/src/'best.pt',map_location='cpu',weights_only=False)
        enc=torch.load(R04/'models'/src/'encoder_best.pt',map_location='cpu',weights_only=False)
        req(best['n01_manifest_sha256']==run04['n01_manifest_sha256'],'Wrong R04 model provenance')
        req(enc['n01_manifest_sha256']==run04['n01_manifest_sha256'],'Wrong encoder provenance')
        req(enc.get('stage')=='R04' and enc.get('source')==src and enc.get('feature_dim')==2048 and enc.get('fc')=='Identity','Wrong R04 exported encoder')
        req(enc.get('signature')==best.get('signature'),'R04 encoder/model signature mismatch')
        req(all(torch.equal(v,best['model']['encoder.'+k]) for k,v in enc['encoder_state'].items()),'R04 exported encoder differs from selected classifier')
        self.encoder.load_state_dict(enc['encoder_state'],strict=True)
        self.classifier.load_state_dict({'weight':best['model']['classifier.weight'],'bias':best['model']['classifier.bias']},strict=True)
    def forward(self,x):
        f=self.encoder(x);return f,self.classifier(f)
encoders={s:R04Classifier(s).to(DEVICE).eval() for s in ['rotation','simclr']}
normalization=R06/'reports/train_only_normalization.npz';req(normalization.is_file(),'Missing R06 feature normalization')
with np.load(normalization,allow_pickle=False) as z:
    stats={s:(np.asarray(z[s+'_mean'],dtype=np.float32),np.asarray(z[s+'_std'],dtype=np.float32)) for s in ['rotation','simclr']}
for s,(m,sd) in stats.items():req(m.shape==(2048,) and sd.shape==(2048,) and np.all(sd>0),'Invalid training-only normalization')

## 3. Four trained R06 fusion architectures and checks

In [4]:
from torch.nn import functional as F
class FusionConcat(nn.Module):
    def __init__(self):
        super().__init__();self.body=nn.Sequential(nn.LayerNorm(4096),nn.Linear(4096,512),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(512,128),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(128,8))
    def forward(self,r,s):return self.body(torch.cat([r,s],dim=1))
class FusionGatedV2(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.gate=nn.Sequential(nn.Linear(512,256),nn.Sigmoid())
        self.classify=nn.Sequential(nn.LayerNorm(256),nn.Dropout(DROPOUT),nn.Linear(256,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);g=self.gate(torch.cat([a,b],1))
        return self.classify(g*a+(1-g)*b)
class FusionGatedV3(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.gate=nn.Linear(512,512)
        self.classify=nn.Sequential(nn.LayerNorm(512),nn.Linear(512,128),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(128,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);g=torch.softmax(self.gate(torch.cat([a,b],1)).reshape(-1,2,256),dim=1)
        mix=g[:,0,:]*a+g[:,1,:]*b
        return self.classify(torch.cat([mix,torch.abs(a-b)],1))
class ResidualFusion(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256))
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256))
        self.res=nn.Sequential(nn.LayerNorm(512),nn.Linear(512,256),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(256,256))
        self.classify=nn.Sequential(nn.LayerNorm(256),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(256,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);return self.classify(.5*(a+b)+self.res(torch.cat([a,b],1)))
MODELS={'FusionConcat':FusionConcat,'FusionGatedV2':FusionGatedV2,'FusionGatedV3':FusionGatedV3,'ResidualFusion':ResidualFusion}

In [5]:
fusions={}
for s in SOURCES[2:]:
    p=R06/'models'/f'{s}_best.pt';req(p.is_file(),f'Missing R06 checkpoint {p}')
    ck=torch.load(p,map_location='cpu',weights_only=False)
    cfg=ck['config'];req(cfg['classes']==CLASSES and cfg['name']==s and cfg['fit_partition']=='train','Wrong trained R06 configuration')
    req(cfg['r05_manifest_sha256']==run04['n01_manifest_sha256'],'R06 checkpoint from different N01 manifest')
    m=MODELS[s]().to(DEVICE).eval();m.load_state_dict(ck['model'],strict=True);fusions[s]=m
print('Fusion checkpoints verified:',list(fusions))

Fusion checkpoints verified: ['FusionConcat', 'FusionGatedV2', 'FusionGatedV3', 'ResidualFusion']


## 4. N01 original image-preprocessing equivalence audit

In [6]:
# Recreate the *actual* N01 3.1.0 processed JPEGs (not approximate resizing).
# Source: lesion-n01-dataset-audit-grouped-splits.py / resize_pad / audit_member.
from PIL import ImageOps
NORM=transforms.Normalize([.485,.456,.406],[.229,.224,.225])
PREPROCESS_VERSION='n01-pipeline-3.1.0-jpeg94-edge-median'

def n01_prepared_jpeg(source_bytes):
    with Image.open(io.BytesIO(source_bytes)) as src:
        src.load();rgb=ImageOps.exif_transpose(src).convert('RGB')
    edges_small=np.asarray(rgb.resize((32,32),Image.Resampling.BILINEAR))
    edges=np.concatenate((edges_small[0],edges_small[-1],edges_small[:,0],edges_small[:,-1]),axis=0)
    color=tuple(np.median(edges,axis=0).astype(np.uint8).tolist())
    padded=ImageOps.pad(rgb,(224,224),method=Image.Resampling.BICUBIC,color=color)
    buf=io.BytesIO();padded.save(buf,format='JPEG',quality=94,subsampling=0,optimize=True)
    return buf.getvalue()

def transform_bytes(data,kind):
    if kind=='n01_exact':data=n01_prepared_jpeg(data)
    else:req(kind=='prepared','Unsupported N01 preprocessing: '+str(kind))
    with Image.open(io.BytesIO(data)) as im:
        req(im.size==(224,224),'Processed N01 image is not 224x224')
        tensor=transforms.ToTensor()(im.convert('RGB'))
    return NORM(tensor)

# Optional source/resized SHA verification when the N01 master is mounted.
masters=[]
for mp in INPUT.rglob('experiment_master_manifest.csv'):
    if mp.is_file() and sha(mp)==run04['n01_manifest_sha256']:masters.append(mp)
req(len(masters)<=1,f'Ambiguous verified N01 manifests: {masters}')
checks_by_id={}
if masters:
    m=pd.read_csv(masters[0],dtype=str,keep_default_na=False)
    req({'image_id','source_sha256','resized_sha256'}.issubset(m.columns),'N01 image checksum fields missing')
    req(m.image_id.is_unique and len(m)==25331,'N01 master identity/count mismatch')
    checks_by_id=m.set_index('image_id')[['source_sha256','resized_sha256']].to_dict('index')
    print('Verifying prepared JPEG checksums against original N01 manifest:',masters[0])
else:print('N01 master not mounted: R04 probability-equivalence check remains mandatory')

ref=R04/'predictions/rotation/validation.npz'
req(ref.is_file(),'Missing R04 reference probabilities for preprocessing verification')
with np.load(ref,allow_pickle=False) as z:
    check_ids=z['image_id'].astype(str);ref_rotation=np.asarray(z['proba'],dtype=np.float64)
with np.load(R04/'predictions/simclr/validation.npz',allow_pickle=False) as z:
    req(np.array_equal(check_ids,z['image_id'].astype(str)),'R04 source validation IDs differ')
    ref_simclr=np.asarray(z['proba'],dtype=np.float64)
indices=np.unique(np.linspace(0,len(check_ids)-1,min(24,len(check_ids)),dtype=int))
errs={};byte_checks=[]
if archive is not None:
    req(all(check_ids[i] in members for i in indices),'R04 validation images absent from official ZIP')
    with zipfile.ZipFile(archive) as z:
        sample=[z.read(members[check_ids[i]]) for i in indices]
    if checks_by_id:
        for ix,data in zip(indices,sample):
            name=check_ids[ix];record=checks_by_id[name]
            raw_sha=hashlib.sha256(data).hexdigest()
            prepared_bytes=n01_prepared_jpeg(data)
            jpeg_sha=hashlib.sha256(prepared_bytes).hexdigest()
            row={'image_id':name,'source_match':raw_sha==record['source_sha256'],
                 'prepared_match':jpeg_sha==record['resized_sha256']}
            byte_checks.append(row)
        pd.DataFrame(byte_checks).to_csv(OUT/'reports/n01_sample_image_sha256_audit.csv',index=False)
        req(all(x['source_match'] for x in byte_checks),'Raw ISIC image bytes disagree with N01 source checksums')
        # Re-encoding JPEG can depend on Pillow/libjpeg version; raw input probabilities are verified below.
        print('N01 prepared JPEG byte-identical:',sum(x['prepared_match'] for x in byte_checks),'/',len(byte_checks))
    with torch.inference_mode():
        x=torch.stack([transform_bytes(blob,'n01_exact') for blob in sample]).to(DEVICE)
        errs['n01_exact']={}
        for src,ref_prob in [('rotation',ref_rotation),('simclr',ref_simclr)]:
            _,logits=encoders[src](x)
            pred=torch.softmax(logits.float(),1).cpu().numpy()
            errs['n01_exact'][src]=float(np.max(np.abs(pred-ref_prob[indices])))
if N01_PREPARED and all(check_ids[i] in N01_PREPARED for i in indices):
    with torch.inference_mode():
        x=torch.stack([transform_bytes(N01_PREPARED[check_ids[i]].read_bytes(),'prepared') for i in indices]).to(DEVICE)
        errs['prepared']={}
        for src,ref_prob in [('rotation',ref_rotation),('simclr',ref_simclr)]:
            _,logits=encoders[src](x)
            pred=torch.softmax(logits.float(),1).cpu().numpy()
            errs['prepared'][src]=float(np.max(np.abs(pred-ref_prob[indices])))
print('R04 exact input reproduction:',json.dumps(errs,indent=2))
req(bool(errs),'No available images for R04 input verification')
PREPROCESS=os.getenv('G02B_PREPROCESS','auto')
if PREPROCESS=='auto':PREPROCESS=min(errs,key=lambda k:max(errs[k].values()))
req(PREPROCESS in errs,f'Unavailable preprocessing mode: {PREPROCESS}; choices={list(errs)}')
worst=max(errs[PREPROCESS].values())
writejson(OUT/'reports/input_reproduction.json',{
    'selected_preprocess':PREPROCESS,'pipeline_version':PREPROCESS_VERSION,
    'max_abs_probability_errors':errs,'worst_error':worst,
    'source_sha_verified':bool(checks_by_id),'sample_image_sha':byte_checks,
    'r04_reference_ids':check_ids[indices].tolist()})
req(worst<.025,f'Exact N01 preprocessing failed R04 probability check: max error={worst:.4f}. '
    'Inspect input_reproduction.json, N01 image checksums and checkpoint identity. No threshold bypass.')
print('N01 preprocessing verified:',PREPROCESS,'worst max probability error:',worst)


N01 master not mounted: R04 probability-equivalence check remains mandatory
R04 exact input reproduction: {
  "n01_exact": {
    "rotation": 0.001303166151046753,
    "simclr": 0.0017627477645874023
  }
}
N01 preprocessing verified: n01_exact worst max probability error: 0.0017627477645874023


## 5. Predict and save six sources in resumable chunks

In [7]:
# Reuse previous identical G02-B chunk files when attached as Kaggle inputs.
for pm in INPUT.rglob('g02b_status.json'):
    prev=json.loads(pm.read_text())
    if prev.get('cohort')!=mode or prev.get('n01_manifest_sha256')!=run04['n01_manifest_sha256'] or prev.get('preprocess')!=PREPROCESS or prev.get('preprocess_version')!=PREPROCESS_VERSION:continue
    for row in prev.get('chunks',[]):
        p=pm.parent.parent/'predictions'/row['file']
        if not p.is_file() or sha(p)!=row['sha256']:continue
        dest=OUT/'predictions'/p.name
        if not dest.exists():shutil.copy2(p,dest)

def batch_predict(items):
    with torch.inference_mode():
        x=torch.stack([transform_bytes(blob,PREPROCESS) for blob in items]).to(DEVICE)
        features={};prob={}
        for src in ['rotation','simclr']:
            f,z=encoders[src](x)
            features[src]=f.float().cpu().numpy()
            prob[src]=torch.softmax(z.float(),1).cpu().numpy().astype(np.float32)
        a=torch.from_numpy(np.clip((features['rotation']-stats['rotation'][0])/stats['rotation'][1],-12,12).astype('float32')).to(DEVICE)
        b=torch.from_numpy(np.clip((features['simclr']-stats['simclr'][0])/stats['simclr'][1],-12,12).astype('float32')).to(DEVICE)
        for name,model in fusions.items():
            prob[name]=torch.softmax(model(a,b).float(),1).cpu().numpy().astype(np.float32)
        return prob

rows=[]
for start in range(0,len(df),CHUNK):
    end=min(start+CHUNK,len(df));part=df.iloc[start:end].reset_index(drop=True)
    token=hashlib.sha256((PREPROCESS_VERSION+'|'+PREPROCESS+'|'+run04['n01_manifest_sha256']+'|'+ '\n'.join(part.image_id)).encode()).hexdigest()[:16]
    target=OUT/'predictions'/f'{mode}_{start:05d}_{end:05d}_{token}.npz'
    if target.exists():
        with np.load(target,allow_pickle=False) as z:
            req(np.array_equal(z['image_id'].astype(str),part.image_id.to_numpy()),'Cached image identities differ')
            req(all('prob__'+s in z for s in SOURCES),'Incomplete cached source')
        print('REUSE',target.name);rows.append({'file':target.name,'sha256':sha(target),'start':start,'end':end});continue
    if time.monotonic()-START>MAX_HOURS*3600:print('TIME GUARD, resumable pause at',start);break
    result={s:[] for s in SOURCES}
    from contextlib import nullcontext
    with (zipfile.ZipFile(archive) if archive is not None else nullcontext(None)) as z:
        for j in range(0,len(part),BATCH):
            names=part.image_id.iloc[j:j+BATCH].tolist()
            if PREPROCESS=='prepared':
                req(all(x in N01_PREPARED for x in names),'Exact N01 prepared file missing from mounted output')
                blobs=[N01_PREPARED[x].read_bytes() for x in names]
            else:blobs=[z.read(members[x]) for x in names]
            p=batch_predict(blobs)
            for s in SOURCES:result[s].append(p[s])
    payload={'image_id':part.image_id.to_numpy(dtype=str),'source':part.source.to_numpy(dtype=str),
             'group_id':part.group_id.fillna('').to_numpy(dtype=str),'y':part.label_id.to_numpy(dtype=np.int32),
             'classes':np.array(CLASSES),'model_names':np.array(SOURCES),'n01_manifest_sha256':np.array(run04['n01_manifest_sha256']),
             'preprocess':np.array(PREPROCESS),'preprocess_version':np.array(PREPROCESS_VERSION)}
    for s in SOURCES:
        a=np.concatenate(result[s],axis=0);req(a.shape==(len(part),8) and np.isfinite(a).all(),'Malformed source predictions')
        req(np.max(abs(a.sum(1)-1))<1e-4,'Unnormalized probabilities');payload['prob__'+s]=a
    np.savez_compressed(target,**payload);rows.append({'file':target.name,'sha256':sha(target),'start':start,'end':end})
    print('SAVED',end,'/',len(df));torch.cuda.empty_cache() if DEVICE.type=='cuda' else None
processed=sum(x['end']-x['start'] for x in rows)
report={'stage':'G02-B','status':'COMPLETED' if processed==len(df) else 'PAUSED',
    'cohort':mode,'images':len(df),'processed':processed,'model_names':SOURCES,'classes':CLASSES,'seed':SEED,
    'n01_manifest_sha256':run04['n01_manifest_sha256'],'preprocess':PREPROCESS,'preprocess_version':PREPROCESS_VERSION,'chunks':rows,
    'note':'No stacker or ensemble fitted; original frozen R04/R06 sources; validation only unless explicit locked-test permission.'}
writejson(OUT/'reports/g02b_status.json',report)
print(report['status'],processed,'/',len(df))

SAVED 256 / 2109
SAVED 512 / 2109
SAVED 768 / 2109
SAVED 1024 / 2109
SAVED 1280 / 2109
SAVED 1536 / 2109
SAVED 1792 / 2109
SAVED 2048 / 2109
SAVED 2109 / 2109
COMPLETED 2109 / 2109


## 6. Source-conditional metrics, eight outputs retained

In [8]:
from sklearn.metrics import balanced_accuracy_score,accuracy_score,f1_score,confusion_matrix
if report['status']=='COMPLETED':
    pred_chunks=[OUT/'predictions'/r['file'] for r in rows]
    ids=[];origins=[];targets=[];mats={s:[] for s in SOURCES}
    for f in pred_chunks:
        with np.load(f,allow_pickle=False) as z:
            ids.extend(z['image_id'].astype(str).tolist());origins.extend(z['source'].astype(str).tolist());targets.extend(z['y'].astype(int).tolist())
            for s in SOURCES:mats[s].append(np.asarray(z['prob__'+s],dtype=np.float32))
    req(ids==df.image_id.tolist(),'Output image order differs from G01')
    y=np.asarray(targets);origins=np.asarray(origins);results=[];recalls=[]
    for s in SOURCES:
        p=np.concatenate(mats[s]);pred=p.argmax(1)
        for origin in ['HAM10000','BCN20000']:
            idx=np.where(origins==origin)[0]
            if not len(idx):continue
            yy=y[idx];pp=pred[idx];present=np.unique(yy)
            results.append({'source':origin,'model':s,'n':len(idx),'represented_classes':','.join(CLASSES[int(c)] for c in present),
                'balanced_accuracy':float(balanced_accuracy_score(yy,pp)) if len(present)>=2 else None,
                'accuracy':float(accuracy_score(yy,pp)),
                'macro_f1_present_classes':float(f1_score(yy,pp,labels=present,average='macro',zero_division=0)),
                'unsupported_prediction_rate':float((~np.isin(pp,present)).mean())})
            cm=confusion_matrix(yy,pp,labels=np.arange(8))
            for c in present:recalls.append({'source':origin,'model':s,'class':CLASSES[int(c)],'recall':float(cm[c,c]/max(1,cm[c].sum())),'n':int(cm[c].sum())})
    pd.DataFrame(results).to_csv(OUT/'tables/per_source_base_models.csv',index=False)
    pd.DataFrame(recalls).to_csv(OUT/'tables/per_class_recall.csv',index=False)
    print(pd.DataFrame(results)[['source','model','n','balanced_accuracy','unsupported_prediction_rate']].to_string(index=False))
else:print('PAUSED — no final metrics until all prediction chunks are complete')

  source          model    n  balanced_accuracy  unsupported_prediction_rate
HAM10000       rotation 1060           0.616183                          0.0
BCN20000       rotation 1049           0.470219                          0.0
HAM10000         simclr 1060           0.676655                          0.0
BCN20000         simclr 1049           0.443982                          0.0
HAM10000   FusionConcat 1060           0.648940                          0.0
BCN20000   FusionConcat 1049           0.467775                          0.0
HAM10000  FusionGatedV2 1060           0.686398                          0.0
BCN20000  FusionGatedV2 1049           0.444905                          0.0
HAM10000  FusionGatedV3 1060           0.666239                          0.0
BCN20000  FusionGatedV3 1049           0.459472                          0.0
HAM10000 ResidualFusion 1060           0.676990                          0.0
BCN20000 ResidualFusion 1049           0.476646                          0.0